In [ ]:
# ============================================================================
# SECTION 1: PACKAGE IMPORT
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("DATA CLEANING AND ANALYSIS PIPELINE STARTED")
print("="*80)
print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

DATA CLEANING AND ANALYSIS PIPELINE STARTED
Start Time: 2026-06-13 04:44:14


In [ ]:
# ============================================================================
# SECTION 2: READ CSV FILE
# ============================================================================
print("\n[STEP 1] Loading the dataset...")
print("-"*50)

# Read the unclean CSV file
df = pd.read_csv('/content/flight_customer_uncleandata.csv')

print(f"Dataset loaded successfully")
print(f"Shape of dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display basic info
print("\nDataset Info:")
print("-"*50)
print(df.info())

print("\nFirst 5 rows preview:")
print("-"*50)
print(df.head())

print("\nLast 5 rows preview:")
print("-"*50)
print(df.tail())



[STEP 1] Loading the dataset...
--------------------------------------------------
Dataset loaded successfully
Shape of dataset: 71 rows x 15 columns
Memory usage: 0.03 MB

Dataset Info:
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MEMBER_NO          71 non-null     int64  
 1   FFP_DATE           71 non-null     object 
 2   FIRST_FLIGHT_DATE  71 non-null     object 
 3   GENDER             71 non-null     object 
 4   WORK_CITY          67 non-null     object 
 5   WORK_PROVINCE      66 non-null     object 
 6   WORK_COUNTRY       71 non-null     object 
 7   AGE                70 non-null     float64
 8   LAST_FLIGHT_DATE   71 non-null     object 
 9   LAST_TO_END        71 non-null     int64  
 10  AVG_INTERVAL       71 non-null     float64
 11  MAX_INTERVAL       71 non-nul

In [ ]:
# ============================================================================
# SECTION 3: EXPLORATORY DATA ANALYSIS (EDA) PROCESS
# ============================================================================
print("\n" + "="*80)
print("[STEP 2] EXPLORATORY DATA ANALYSIS (EDA)")
print("="*80)

# 3.1 UNDERSTAND VARIABLES
print("\n[3.1] Understanding Variables:")
print("-"*50)
print(f"Total columns: {len(df.columns)}")
print(f"\nColumn names and data types:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:25s} -> {df[col].dtype}")

# 3.2 SUMMARY STATISTICS
print("\n[3.2] Summary Statistics (Numerical Columns):")
print("-"*50)
print(df.describe())

print("\n[3.2] Summary Statistics (Categorical Columns):")
print("-"*50)
print(df.describe(include=['object']))

# 3.3 CHECK MISSING VALUES
print("\n[3.3] Missing Values Analysis:")
print("-"*50)
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100,
    'Data_Type': df.dtypes.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)
print(missing_df.to_string(index=False))

# 3.4 UNIQUE VALUE ANALYSIS
print("\n[3.4] Unique Value Analysis (Categorical Columns):")
print("-"*50)
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f"  {col:25s}: {unique_count:5d} unique values")
    if unique_count <= 10:
        unique_vals = df[col].unique()[:10]
        print(f"    Values: {unique_vals}")

# 3.5 DATA TYPES ANALYSIS - FIXED VERSION
print("\n[3.5] Data Type Analysis:")
print("-"*50)
dtype_counts = df.dtypes.value_counts()
print("Data Type Distribution:")
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count} columns")

# Create a summary DataFrame for data types
dtype_summary = []
for col in df.columns:
    dtype_summary.append({
        'Column_Name': col,
        'Current_Data_Type': str(df[col].dtype),
        'Unique_Values': df[col].nunique(),
        'Null_Count': df[col].isnull().sum(),
        'Null_Percentage': (df[col].isnull().sum() / len(df)) * 100
    })
dtype_df = pd.DataFrame(dtype_summary)
print("\nDetailed Column Information:")
print(dtype_df.to_string(index=False))



[STEP 2] EXPLORATORY DATA ANALYSIS (EDA)

[3.1] Understanding Variables:
--------------------------------------------------
Total columns: 15

Column names and data types:
   1. MEMBER_NO                 -> int64
   2. FFP_DATE                  -> object
   3. FIRST_FLIGHT_DATE         -> object
   4. GENDER                    -> object
   5. WORK_CITY                 -> object
   6. WORK_PROVINCE             -> object
   7. WORK_COUNTRY              -> object
   8. AGE                       -> float64
   9. LAST_FLIGHT_DATE          -> object
  10. LAST_TO_END               -> int64
  11. AVG_INTERVAL              -> float64
  12. MAX_INTERVAL              -> int64
  13. EXCHANGE_COUNT            -> int64
  14. Points_Sum                -> int64
  15. Point_NotFlight           -> int64

[3.2] Summary Statistics (Numerical Columns):
--------------------------------------------------
          MEMBER_NO        AGE  LAST_TO_END  AVG_INTERVAL  MAX_INTERVAL  \
count     71.000000  70.0000

In [ ]:
# ============================================================================
# SECTION 4: STATISTICAL ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("[STEP 3] STATISTICAL DATA ANALYSIS")
print("="*80)

# 4.1 CENTRAL TENDENCY AND DISPERSION
print("\n[4.1] Central Tendency and Dispersion:")
print("-"*50)

# Separate numerical columns for analysis
numerical_cols = df.select_dtypes(include=[np.number]).columns

for col in numerical_cols:
    try:
        # Convert to numeric, coercing errors
        valid_data = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(valid_data) > 0:
            print(f"\nColumn: {col.upper()}")
            print(f"   Mean     : {valid_data.mean():.2f}")
            print(f"   Median   : {valid_data.median():.2f}")
            print(f"   Mode     : {valid_data.mode().iloc[0] if len(valid_data.mode()) > 0 else 'N/A'}")
            print(f"   Std Dev  : {valid_data.std():.2f}")
            print(f"   Variance : {valid_data.var():.2f}")
            print(f"   Min      : {valid_data.min():.2f}")
            print(f"   Max      : {valid_data.max():.2f}")
            print(f"   Range    : {valid_data.max() - valid_data.min():.2f}")
            print(f"   Q1 (25th): {valid_data.quantile(0.25):.2f}")
            print(f"   Q3 (75th): {valid_data.quantile(0.75):.2f}")
            print(f"   IQR      : {valid_data.quantile(0.75) - valid_data.quantile(0.25):.2f}")
            print(f"   Skewness : {valid_data.skew():.2f}")
            print(f"   Kurtosis : {valid_data.kurtosis():.2f}")
    except Exception as e:
        continue

# 4.2 CORRELATION MATRIX
print("\n[4.2] Correlation Analysis:")
print("-"*50)
# Convert all numerical columns properly
numerical_data = df[numerical_cols].apply(pd.to_numeric, errors='coerce')
correlation_matrix = numerical_data.corr()
print("Correlation Matrix:")
print(correlation_matrix.to_string())

# Identify highly correlated features
print("\nHighly Correlated Features (correlation > 0.5):")
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.5:
            high_corr.append((correlation_matrix.columns[i], correlation_matrix.columns[j],
                            correlation_matrix.iloc[i, j]))
            print(f"  {correlation_matrix.columns[i]} <-> {correlation_matrix.columns[j]}: {correlation_matrix.iloc[i, j]:.3f}")

if len(high_corr) == 0:
    print("  No highly correlated features found")


[STEP 3] STATISTICAL DATA ANALYSIS

[4.1] Central Tendency and Dispersion:
--------------------------------------------------

Column: MEMBER_NO
   Mean     : 35821.45
   Median   : 38241.00
   Mode     : 916
   Std Dev  : 17401.97
   Variance : 302828518.51
   Min      : 916.00
   Max      : 62751.00
   Range    : 61835.00
   Q1 (25th): 23079.50
   Q3 (75th): 49472.50
   IQR      : 26393.00
   Skewness : -0.35
   Kurtosis : -0.94

Column: AGE
   Mean     : 48.59
   Median   : 48.00
   Mode     : 48.0
   Std Dev  : 9.42
   Variance : 88.77
   Min      : 30.00
   Max      : 76.00
   Range    : 46.00
   Q1 (25th): 42.25
   Q3 (75th): 53.00
   IQR      : 10.75
   Skewness : 0.49
   Kurtosis : 0.37

Column: LAST_TO_END
   Mean     : 24.01
   Median   : 8.00
   Mode     : 1
   Std Dev  : 43.59
   Variance : 1900.30
   Min      : 1.00
   Max      : 304.00
   Range    : 303.00
   Q1 (25th): 3.00
   Q3 (75th): 22.00
   IQR      : 19.00
   Skewness : 4.33
   Kurtosis : 24.34

Column: AVG_INTER

In [ ]:
# ============================================================================
# SECTION 5: DATA DUPLICATE ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("[STEP 4] DUPLICATE DATA ANALYSIS")
print("="*80)

print(f"\n[4.1] Total rows before duplicate check: {len(df)}")
print(f"[4.2] Duplicate rows found: {df.duplicated().sum()}")

# Identify duplicate rows based on all columns
duplicate_rows = df[df.duplicated(keep='first')]
if len(duplicate_rows) > 0:
    print(f"\n[4.3] Sample of duplicate rows (first 5):")
    print(duplicate_rows.head())

# Check duplicates based on property_id
print(f"\n[4.4] Duplicates based on MEMBER_NO:")
print(f"  Unique member no: {df['MEMBER_NO'].nunique()}")
print(f"  Duplicate member no: {df['MEMBER_NO'].duplicated().sum()}")

# ============================================================================
# SECTION 6: DATA NULL ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("[STEP 5] NULL VALUE ANALYSIS")
print("="*80)

print(f"\n[5.1] Total missing values in dataset: {df.isnull().sum().sum()}")
print(f"[5.2] Percentage of missing values: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")

print("\n[5.3] Column-wise null analysis:")
null_analysis = pd.DataFrame({
    'Column': df.columns,
    'Null_Count': df.isnull().sum(),
    'Null_Percentage': (df.isnull().sum() / len(df)) * 100,
    'Non_Null_Count': df.notnull().sum()
})
null_analysis = null_analysis[null_analysis['Null_Count'] > 0].sort_values('Null_Percentage', ascending=False)
print(null_analysis.to_string(index=False))


[STEP 4] DUPLICATE DATA ANALYSIS

[4.1] Total rows before duplicate check: 71
[4.2] Duplicate rows found: 0

[4.4] Duplicates based on MEMBER_NO:
  Unique member no: 71
  Duplicate member no: 0

[STEP 5] NULL VALUE ANALYSIS

[5.1] Total missing values in dataset: 10
[5.2] Percentage of missing values: 0.94%

[5.3] Column-wise null analysis:
       Column  Null_Count  Null_Percentage  Non_Null_Count
WORK_PROVINCE           5         7.042254              66
    WORK_CITY           4         5.633803              67
          AGE           1         1.408451              70


In [ ]:
# ============================================================================
# SECTION 7: DELETE DUPLICATES
# ============================================================================
print("\n" + "="*80)
print("[STEP 6] DELETING DUPLICATES")
print("="*80)

print(f"\n[6.1] Before deletion: {len(df)} rows")
df = df.drop_duplicates(keep='first')
print(f"[6.2] After deletion: {len(df)} rows")
print(f"[6.3] Total duplicates removed: {df.duplicated().sum()}")

# Reset index after deletion
df = df.reset_index(drop=True)
print(f"[6.4] Index reset successfully")


[STEP 6] DELETING DUPLICATES

[6.1] Before deletion: 71 rows
[6.2] After deletion: 71 rows
[6.3] Total duplicates removed: 0
[6.4] Index reset successfully


In [ ]:
# ============================================================================
# SECTION 8: FILLING NULL VALUES
# ============================================================================
print("\n" + "="*80)
print("[STEP 7] HANDLING NULL VALUES")
print("="*80)

# Create a copy for analysis before cleaning
df_original = df.copy()

# 8.1 Handle numerical columns
print("\n[7.1] Handling Numerical Columns:")
print("-"*50)

# Identify numerical columns that are stored as object
for col in df.columns:
    if col in ['MEMBER_NO', 'AGE', 'LAST_TO_END', 'AVG_INTERVAL', 'MAX_INTERVAL',
               'Points_Sum', 'EXCHANGE_COUNT', 'Point_NotFlight']:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            print(f"  Converted {col} to numeric")
        except:
            pass

# Update numerical columns list
numerical_cols = df.select_dtypes(include=[np.number]).columns

for col in numerical_cols:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        print(f"\n  Column: {col}")
        print(f"    Null values: {null_count} ({null_count/len(df)*100:.2f}%)")

        # Fill based on data distribution
        if len(df[col].dropna()) > 0:
            if df[col].skew() > 1 or df[col].skew() < -1:
                # Highly skewed - use median
                fill_value = df[col].median()
                method = "MEDIAN (skewed distribution)"
            else:
                # Normal-ish - use mean
                fill_value = df[col].mean()
                method = "MEAN (normal distribution)"

            df[col].fillna(fill_value, inplace=True)
            print(f"    Filled with: {method} = {fill_value:.2f}")

# 8.2 Handle categorical columns
print("\n[7.2] Handling Categorical Columns:")
print("-"*50)

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        print(f"\n  Column: {col}")
        print(f"    Null values: {null_count} ({null_count/len(df)*100:.2f}%)")

        # Fill with mode (most frequent value)
        mode_value = df[col].mode()
        if len(mode_value) > 0:
            fill_value = mode_value[0]
            df[col].fillna(fill_value, inplace=True)
            print(f"    Filled with: MODE = '{fill_value}'")
        else:
            df[col].fillna("Unknown", inplace=True)
            print(f"    Filled with: 'Unknown'")

# 8.3 Handle remaining nulls
print("\n[7.3] Final Null Check:")
print("-"*50)
remaining_nulls = df.isnull().sum().sum()
if remaining_nulls > 0:
    print(f"Warning: {remaining_nulls} null values remain")
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print(f"All null values have been handled successfully")


[STEP 7] HANDLING NULL VALUES

[7.1] Handling Numerical Columns:
--------------------------------------------------
  Converted MEMBER_NO to numeric
  Converted AGE to numeric
  Converted LAST_TO_END to numeric
  Converted AVG_INTERVAL to numeric
  Converted MAX_INTERVAL to numeric
  Converted EXCHANGE_COUNT to numeric
  Converted Points_Sum to numeric
  Converted Point_NotFlight to numeric

  Column: AGE
    Null values: 1 (1.41%)
    Filled with: MEAN (normal distribution) = 48.59

[7.2] Handling Categorical Columns:
--------------------------------------------------

  Column: WORK_CITY
    Null values: 4 (5.63%)
    Filled with: MODE = 'guangzhou'

  Column: WORK_PROVINCE
    Null values: 5 (7.04%)
    Filled with: MODE = 'guangdong'

[7.3] Final Null Check:
--------------------------------------------------
All null values have been handled successfully


In [ ]:
# ============================================================================
# SECTION 9: DATA TRANSFORMATION
# ============================================================================
print("\n" + "="*80)
print("[STEP 8] DATA TRANSFORMATION")
print("="*80)

# 9.1 Standardize text columns
print("\n[8.1] Standardizing Text Columns:")
print("-"*50)

# province standardization
province_mapping = {
    'guangdong': 'Guangdong', 'GUANGDONG': 'Guangdong', 'Guangdong': 'Guangdong',
    'beijing': 'Beijing', 'BEIJING': 'Beijing', 'Beijing': 'Beijing',
    'zhejiang': 'Zhejiang', 'ZHEJIANG': 'Zhejiang', 'Zhejiang': 'Zhejiang',
    'paris': 'Paris', 'PARIS': 'Paris', 'Paris': 'Paris',
    'xinjiang': 'Xinjiang', 'XINJIANG': 'Xinjiang', 'Xinjiang': 'Xinjiang',
    'guizhou': 'Guizhou', 'GUIZHOU': 'Guizhou', 'Guizhou': 'Guizhou',
    '.': np.nan
}
df['WORK_PROVINCE'] = df['WORK_PROVINCE'].astype(str).str.lower().map(province_mapping).fillna(df['WORK_PROVINCE'])
print("WORK_PROVINCE standardized")

# Property type standardization
city_mapping = {
    'beijing': 'Beijing', 'beijing': 'Beijing',
    'guangzhou': 'Guangzhou', 'GAUNGZHOU': 'Guangzhou',
    'wenzhou': 'Wenzhou', 'WENZHOU': 'Wenzhou',
    'wenzhoushi': 'Wenzhou', 'WENZHOUSHI': 'Wenzhou',
    'wulumuqi': 'Urumqi', 'Wulumuqi': 'Urumqi',
    'wulumuqishi': 'Urumqi', 'Wulumuqishi': 'Urumqi',
    'guiyang': 'Guiyang', 'GUIYANG': 'Guiyang',
    'dongguan': 'Dongguan','DONGGUAN': 'Dongguan',
    'paris': 'Paris', 'PARIS': 'Paris', 'Paris': 'Paris',
    '.': np.nan
}
df['WORK_CITY'] = df['WORK_CITY'].astype(str).str.lower().map(city_mapping).fillna(df['WORK_CITY'])
print("WORK_CITY standardized")

# 8.2 Convert date columns
print("\n[8.2] Converting Date Columns:")
print("-"*50)
try:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    print("Date column converted to datetime format")
    print(f"Date range: {df['date'].min()} to {df['date'].max()}")
except Exception as e:
    print(f"Date conversion warning: {e}")

# 8.3 Create derived features
print("\n[8.3] Creating Derived Features:")
print("-"*50)

# Customer age category
df['AGE_CATEGORY'] = pd.cut(
    df['AGE'],
    bins=[0, 25, 35, 50, 65, float('inf')],
    labels=['Young Adult','Adult','Middle Aged','Senior', 'Elderly']
)
print("Age category created")

# Flight activity category
df['FLIGHT_ACTIVITY'] = pd.cut(
    df['EXCHANGE_COUNT'],
    bins=[-1, 5, 15, 30, float('inf')],
    labels=['Low Activity', 'Moderate Activity', 'High Activity','Very High Activity']
)
print("Flight activity category created")

# Loyalty points category
df['POINTS_CATEGORY'] = pd.cut(
    df['Points_Sum'],
    bins=[0, 100000, 300000, 500000, float('inf')],
    labels=['Low Points', 'Medium Points','High Points','Premium Points']
)
print("Points category created")
print("\nData transformation completed successfully")


[STEP 8] DATA TRANSFORMATION

[8.1] Standardizing Text Columns:
--------------------------------------------------
WORK_PROVINCE standardized
WORK_CITY standardized

[8.2] Converting Date Columns:
--------------------------------------------------
Date conversion warning: 'date'

[8.3] Creating Derived Features:
--------------------------------------------------
Age category created
Flight activity category created
Points category created

Data transformation completed successfully


In [ ]:
# ============================================================================
# SECTION 10: DIFFERENT GRAPH CREATION AND SAVE AS IMAGE
# ============================================================================
print("\n" + "="*80)
print("[STEP 9] GENERATING VISUALIZATION DASHBOARD")
print("="*80)

# Create directory for images
import os
if not os.path.exists('dashboard_images'):
    os.makedirs('dashboard_images')
    print("Created directory: dashboard_images/")

# 10.1 Distribution Plot - Analysis
print("\n[9.1] Generating Distribution Plot...")
fig, axes = plt.subplots(2, 2, figsize=(15, 12)) # Increased figsize for better layout
fig.suptitle('AIRLINE ANALYSIS DASHBOARD', fontsize=16, fontweight='bold')

# Plot 1: Histogram of AGE
axes[0,0].hist(df['AGE'].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0,0].set_xlabel('Age')
axes[0,0].set_ylabel('Number of Customers')
axes[0,0].set_title('Age Distribution')
axes[0,0].axvline(df['AGE'].mean(), color='red', linestyle='--', label=f'Mean: {df["AGE"].mean():.2f}')
axes[0,0].axvline(df['AGE'].median(), color='green', linestyle='--', label=f'Median: {df["AGE"].median():.2f}')
axes[0,0].legend()

# Plot 2: Pie chart for POINTS_CATEGORY
points_category_counts = df['POINTS_CATEGORY'].value_counts()
axes[0,1].pie(points_category_counts.values, labels=points_category_counts.index, autopct='%1.1f%%', startangle=90)
axes[0,1].set_title('Customer Points Category Distribution')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('dashboard_images/01_analysis_dashboard.png', dpi=300, bbox_inches='tight')
plt.close()
print("  Saved: 01_analysis_dashboard.png")

# 10.2 Missing Values Heatmap
print("\n[9.2] Generating Missing Values Heatmap...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('DATA QUALITY DASHBOARD', fontsize=16, fontweight='bold')

# Heatmap of missing values
sns.heatmap(df_original.isnull(), yticklabels=False, cbar=True, ax=axes[0], cmap='YlOrRd')
axes[0].set_title('Missing Values Pattern (Before Cleaning)')
axes[0].set_xlabel('Columns')

# Missing values percentage bar chart
missing_before = (df_original.isnull().sum() / len(df_original) * 100).sort_values(ascending=False)
missing_before = missing_before[missing_before > 0]
if len(missing_before) > 0:
    sns.barplot(x=missing_before.values, y=missing_before.index, ax=axes[1], color='tomato') # Using seaborn barplot for better integration
    axes[1].set_xlabel('Missing Percentage (%)')
    axes[1].set_ylabel('Columns')
    axes[1].set_title('Missing Values Percentage by Column')
else:
    axes[1].text(0.5, 0.5, 'No Missing Values Found', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout
plt.savefig('dashboard_images/02_data_quality_dashboard.png', dpi=300, bbox_inches='tight')
plt.close()
print("  Saved: 02_data_quality_dashboard.png")
print("\nAll visualizations generated and saved successfully")


[STEP 9] GENERATING VISUALIZATION DASHBOARD

[9.1] Generating Distribution Plot...
  Saved: 01_analysis_dashboard.png

[9.2] Generating Missing Values Heatmap...
  Saved: 02_data_quality_dashboard.png

All visualizations generated and saved successfully


In [ ]:
# ============================================================================
# SECTION 11: GENERATE EXCEL REPORT
# ============================================================================
print("\n" + "="*80)
print("[STEP 10] GENERATING EXCEL REPORT")
print("="*80)

# Create Excel writer
excel_filename = f'airlin_customer_data_analysis_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:

    # Sheet 1: Raw vs Cleaned Comparison
    print("\n[10.1] Creating Raw vs Cleaned Comparison Sheet...")
    comparison_df = pd.DataFrame({
        'Metric': ['Total Rows', 'Total Columns', 'Duplicate Rows', 'Total Null Values',
                   'Memory Usage (MB)'],
        'Before Cleaning': [
            df_original.shape[0], df_original.shape[1],
            df_original.duplicated().sum(), df_original.isnull().sum().sum(),
            df_original.memory_usage(deep=True).sum() / 1024**2
        ],
        'After Cleaning': [
            df.shape[0], df.shape[1],
            df.duplicated().sum(), df.isnull().sum().sum(),
            df.memory_usage(deep=True).sum() / 1024**2
        ]
    })
    comparison_df.to_excel(writer, sheet_name='1_Data_Comparison', index=False)
    print("  Sheet created: 1_Data_Comparison")


    # Sheet 2: Price Analysis by Category
    print("\n[10.4] Creating country Analysis Sheet...")
    if 'WORK_COUNTRY' in df.columns and 'Points_Sum' in df.columns:
        country_analysis = df.groupby('WORK_COUNTRY').agg({
            'Points_Sum': ['mean', 'median', 'count', 'std']
        }).round(2)
        country_analysis.columns = ['Avg_Points', 'Total_Points', 'Member_Count', 'Avg_Age']
        country_analysis.to_excel(writer, sheet_name='4_Country_Analysis')
        print("  Sheet created: 2_Country_Analysis")
    else:
        print("  Skipping Country_Analysis")

    # Sheet 3: Gender Analysis
    print("\n[10.5] Creating Gender Analysis Sheet...")
    if 'GENDER' in df.columns and 'Points_Sum' in df.columns:
        gender_analysis = df.groupby('GENDER').agg({
            'Points_Sum': ['mean', 'median', 'count'],
            'AVG_INTERVAL': 'mean'
        }).round(2)
        gender_analysis.columns = ['Avg_Points', 'Total_Points', 'Member_Count', 'Avg_Flight_Interval']
        gender_analysis.to_excel(writer, sheet_name='5_Gender_Analysis')
        print("  Sheet created: 3_Gender_Analysis")
    else:
        print("  Skipping Gender_Analysis")

    # Sheet 4: Cleaned Data
    print("\n[10.6] Creating Cleaned Dataset Sheet...")
    df.to_excel(writer, sheet_name='6_Cleaned_Data', index=False)
    print("  Sheet created: 4_Cleaned_Data")

    # Sheet 7: Data Dictionary - FIXED VERSION
    print("\n[10.7] Creating Data Dictionary Sheet...")

    # Create descriptions for each column
    descriptions = []
    for col in df.columns:
        if col == 'MEMBER_NO':
            desc = 'Unique member ID'
        elif col == 'FFP_DATE':
            desc = 'Frequent flyer program join date'
        elif col == 'FIRST_FLIGHT_DATE':
            desc = 'First flight date'
        elif col == 'GENDER':
            desc = 'Gender of member'
        elif col == 'WORK_CITY':
            desc = 'Work city'
        elif col == 'WORK_PROVINCE':
            desc = 'Work province/state'
        elif col == 'WORK_COUNTRY':
            desc = 'Work country'
        elif col == 'AGE':
            desc = 'Age of member'
        elif col == 'LAST_FLIGHT_DATE':
            desc = 'Last flight date'
        elif col == 'LAST_TO_END':
            desc = 'Days since last flight'
        elif col == 'AVG_INTERVAL':
            desc = 'Average flight interval'
        elif col == 'MAX_INTERVAL':
            desc = 'Maximum flight interval'
        elif col == 'EXCHANGE_COUNT':
            desc = 'Points redemption count'
        elif col == 'Points_Sum':
            desc = 'Total loyalty points'
        elif col == 'Point_NotFlight':
            desc = 'Points not earned from flights'
        else:
            desc = 'Derived or calculated field'

        descriptions.append(desc)

    data_dict = pd.DataFrame({
        'Column_Name': df.columns,
        'Data_Type': [str(dtype) for dtype in df.dtypes.values],
        'Description': descriptions,
        'Unique_Values': [df[col].nunique() for col in df.columns],
        'Null_Count': [df[col].isnull().sum() for col in df.columns],
        'Null_Percentage': [(df[col].isnull().sum() / len(df)) * 100 for col in df.columns]
    })
    data_dict.to_excel(writer, sheet_name='7_Data_Dictionary', index=False)
    print("  Sheet created: 7_Data_Dictionary")

print(f"\nExcel report generated successfully")
print(f"Filename: {excel_filename}")


[STEP 10] GENERATING EXCEL REPORT

[10.1] Creating Raw vs Cleaned Comparison Sheet...
  Sheet created: 1_Data_Comparison

[10.4] Creating country Analysis Sheet...
  Sheet created: 2_Country_Analysis

[10.5] Creating Gender Analysis Sheet...
  Sheet created: 3_Gender_Analysis

[10.6] Creating Cleaned Dataset Sheet...
  Sheet created: 4_Cleaned_Data

[10.7] Creating Data Dictionary Sheet...
  Sheet created: 7_Data_Dictionary

Excel report generated successfully
Filename: airlin_customer_data_analysis_report_20260613_044417.xlsx


In [ ]:
# ============================================================================
# SECTION 12: FINAL SUMMARY REPORT
# ============================================================================
print("\n" + "="*80)
print("FINAL EXECUTION SUMMARY")
print("="*80)

print(f"""
DATA CLEANING COMPLETED

Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
End Time:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

DATA QUALITY METRICS:
- Original Dataset Size: {df_original.shape[0]} rows x {df_original.shape[1]} columns
- Cleaned Dataset Size:  {df.shape[0]} rows x {df.shape[1]} columns
- Duplicates Removed:    {df_original.duplicated().sum()} rows
- Null Values Filled:    {df_original.isnull().sum().sum()} cells

OUTPUT FILES GENERATED:
- Dashboard Images: {len([f for f in os.listdir('dashboard_images') if f.endswith('.png')])} files in 'dashboard_images/'
- Excel Report:     {excel_filename}

All tasks completed successfully
""")

print("\nANALYSIS COMPLETE")
print("You can now:")
print("  1. Check the 'dashboard_images/' folder for visualizations")
print("  2. Open the Excel report for detailed analysis")
print("  3. Use the cleaned dataset for further modeling")
print("\n" + "="*80)
print("END OF ANALYSIS PIPELINE")
print("="*80)


FINAL EXECUTION SUMMARY

DATA CLEANING COMPLETED

Start Time: 2026-06-13 04:44:17
End Time:   2026-06-13 04:44:17

DATA QUALITY METRICS:
- Original Dataset Size: 71 rows x 15 columns
- Cleaned Dataset Size:  71 rows x 18 columns
- Duplicates Removed:    0 rows
- Null Values Filled:    10 cells

OUTPUT FILES GENERATED:
- Dashboard Images: 2 files in 'dashboard_images/'
- Excel Report:     airlin_customer_data_analysis_report_20260613_044417.xlsx

All tasks completed successfully


ANALYSIS COMPLETE
You can now:
  1. Check the 'dashboard_images/' folder for visualizations
  2. Open the Excel report for detailed analysis
  3. Use the cleaned dataset for further modeling

END OF ANALYSIS PIPELINE
